<a href="https://colab.research.google.com/github/Solo7602/IIAS/blob/lab4/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer
import re
import nltk
from razdel import tokenize

nltk.download('stopwords')

# Загрузка данных
data = pd.read_csv('sample_data/10k_dataset_processed_final.csv')

morph = MorphAnalyzer()

patterns = "[A-Za-z0-9!#$%&'()*+,./:;<=>?@[\]^_`{|}~—\"\-]+"

russian_stopwords = stopwords.words('russian')

def preprocess_text(text):
    text = re.sub(patterns, ' ', text)
    text = str(text).lower()
    words = [token.text for token in tokenize(text)]
    words = [
        morph.parse(word)[0].normal_form
        for word in words
        if word not in russian_stopwords
    ]
    return ' '.join(words)

data['Comment_processed'] = data['Comment'].apply(preprocess_text)

X = data['Comment_processed']
y = data['Age']


# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Конвейер для Ridge
pipeline_ridge = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=russian_stopwords)),
    ('ridge', Ridge())
])

# Конвейер для Random Forest
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=russian_stopwords)),
    ('rf', RandomForestRegressor(random_state=42))
])

# Конвейер для XGBoost
pipeline_xgb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=russian_stopwords)),
    ('xgb', XGBRegressor(random_state=42))
])

# Параметры для GridSearch
params_ridge = {'ridge__alpha': [0.01, 0.1, 1, 10, 100]}
params_rf = {'rf__n_estimators': [50, 500], 'rf__max_depth': [10, 20, None], 'rf__max_features': ['sqrt', 'log2'], 'rf__min_samples_split': [2, 5], 'rf__min_samples_leaf': [1, 2]}
params_xgb = {'xgb__n_estimators': [50, 100], 'xgb__learning_rate': [0.01, 0.1]}

scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'RMSE': make_scorer(mean_squared_error),
    'R2': make_scorer(r2_score)
}

def evaluate_model(model, X, y, model_name):
    # Кросс-валидация
    cv_results = cross_validate(model, X, y, cv=5, scoring=scoring)

    # Расчет метрик
    mean_mae = np.mean(cv_results['test_MAE'])
    std_mae = np.std(cv_results['test_MAE'])
    mean_r2 = np.mean(cv_results['test_R2'])
    std_r2 = np.std(cv_results['test_R2'])

    # Вывод результатов
    print(f"\n{model_name} Cross-Validation Results:")
    print(f"MAE: {mean_mae:.3f} ± {std_mae:.3f}")
    print(f"R2: {mean_r2:.3f} ± {std_r2:.3f}")

    # Оценка обученности
    if mean_r2 > 0.7:
        print("Вывод: Модель хорошо обучена (R2 > 0.7)")
    elif mean_r2 > 0.5:
        print("Вывод: Модель удовлетворительная (0.5 < R2 ≤ 0.7)")
    else:
        print("Вывод: Модель плохо обучена (R2 ≤ 0.5)")

# Обучение и оценка Ridge
grid_ridge = GridSearchCV(pipeline_ridge, params_ridge, cv=3, scoring='neg_mean_absolute_error')
grid_ridge.fit(X_train, y_train)
y_pred = grid_ridge.predict(X_test)

print("="*50)
evaluate_model(grid_ridge.best_estimator_, X_train, y_train, "Ridge Regression")

# # Обучение и оценка Random Forest
grid_rf = GridSearchCV(pipeline_rf, params_rf, cv=3, scoring='neg_mean_absolute_error')
grid_rf.fit(X_train, y_train)
y_pred = grid_rf.predict(X_test)

print("="*50)
evaluate_model(grid_rf.best_estimator_, X_train, y_train, "Random Forest")

# # Обучение и оценка XGBoost
grid_xgb = GridSearchCV(pipeline_xgb, params_xgb, cv=3, scoring='neg_mean_absolute_error')
grid_xgb.fit(X_train, y_train)
y_pred = grid_xgb.predict(X_test)

print("="*50)
evaluate_model(grid_xgb.best_estimator_, X_train, y_train, "XGBoost")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Ridge Regression Cross-Validation Results:
MAE: 11.323 ± 0.055
R2: 0.171 ± 0.005
Вывод: Модель плохо обучена (R2 ≤ 0.5)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.9 MB/s eta 0:00:00


# Новый раздел